# Aula 5 — Arquivos e formatos: lendo e gravando dados

> **Objetivo:** ler arquivos CSV e JSON, processar o conteúdo e salvar o resultado.

---

## O problema

Dados chegam de algum lugar — um arquivo CSV num bucket S3, um JSON de uma API, um log de texto. Antes de qualquer transformação, você precisa **ler esse arquivo em Python**.

## 1. Criando arquivos de teste

Vamos criar os arquivos que vamos usar nesta aula:

In [1]:
import os

# Criar pasta de trabalho
os.makedirs('dados', exist_ok=True)

# Criar CSV de exemplo
csv_conteudo = """id,nome,cidade,valor,data
1,Ana Lima,São Paulo,1500.50,15/01/2024
2,Bruno Costa,Rio de Janeiro,800.00,16/01/2024
3,Carla Dias,Belo Horizonte,2200.75,17/01/2024
4,Diego Nunes,Porto Alegre,450.00,18/01/2024
5,Eva Melo,Recife,,19/01/2024
"""

with open('dados/clientes.csv', 'w', encoding='utf-8') as f:
    f.write(csv_conteudo)

print('Arquivo CSV criado: dados/clientes.csv')

Arquivo CSV criado: dados/clientes.csv


**📝 O que essa célula faz:**

Cria uma pasta no disco e escreve um arquivo CSV de teste dentro dela.

```python
os.makedirs('dados', exist_ok=True)

with open('dados/clientes.csv', 'w', encoding='utf-8') as f:
    f.write(csv_conteudo)
```

**📁 `os.makedirs('dados', exist_ok=True)`** → cria a pasta chamada `dados`. O parâmetro `exist_ok=True` diz: "se essa pasta já existir, não dê erro, apenas continue" — sem ele, rodar essa célula uma segunda vez geraria um erro reclamando que a pasta já existe.

**📂 `with open(caminho, 'w', encoding='utf-8') as f:`**

- `open(...)` → abre (ou cria) um arquivo no caminho informado
- `'w'` → modo **escrita** (write); se o arquivo já existir, ele será **sobrescrito** do zero
- `encoding='utf-8'` → garante que acentos e caracteres especiais (como `ã`, `ç`, `é`) sejam salvos corretamente
- `as f` → guarda o arquivo aberto na variável `f`, para usar dentro do bloco
- `with ... :` → garante que o arquivo será **fechado automaticamente** ao final do bloco, mesmo se algo der errado no meio do caminho. É a forma correta e segura de trabalhar com arquivos em Python.

**✍️ `f.write(csv_conteudo)`** → escreve o texto inteiro (guardado na variável `csv_conteudo`, definida acima da célula) dentro do arquivo.

In [2]:
import json

# Criar JSON de exemplo (metadados de pipeline)
metadados = {
    'pipeline': 'ingestao_clientes',
    'versao': '1.2.0',
    'origem': 'crm_salesforce',
    'destino': 's3://lake/bronze/clientes/',
    'configuracao': {
        'batch_size': 10000,
        'timeout_segundos': 300,
        'retry_max': 3
    },
    'tags': ['clientes', 'crm', 'bronze']
}

with open('dados/pipeline_config.json', 'w', encoding='utf-8') as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)

print('Arquivo JSON criado: dados/pipeline_config.json')

Arquivo JSON criado: dados/pipeline_config.json


**📝 O que essa célula faz:**

Cria um dicionário de metadados e o salva como arquivo JSON no disco.

```python
with open('dados/pipeline_config.json', 'w', encoding='utf-8') as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)
```

**💾 `json.dump(dado, arquivo, ...)`** → converte um dicionário Python em texto no formato JSON e escreve diretamente no arquivo aberto (`f`). É diferente de `json.dumps()` (com `s` no final), que devolveria o JSON como texto na memória em vez de já salvar em arquivo.

**🔤 `ensure_ascii=False`** → sem essa opção, o `json.dump` converteria acentos em códigos estranhos como `\u00e3` em vez de mostrar `ã` diretamente no arquivo. Com `ensure_ascii=False`, os caracteres acentuados ficam legíveis no arquivo final.

**📐 `indent=2`** → formata o JSON com **2 espaços de indentação**, deixando o arquivo legível para humanos (cada campo aninhado em uma linha própria, com recuo). Sem essa opção, o JSON seria salvo todo em uma única linha — funcional, mas difícil de ler.

## 2. Lendo arquivos texto e CSV

In [3]:
# Lendo arquivo texto simples
with open('dados/clientes.csv', 'r', encoding='utf-8') as f:
    conteudo = f.read()

print(conteudo)

id,nome,cidade,valor,data
1,Ana Lima,São Paulo,1500.50,15/01/2024
2,Bruno Costa,Rio de Janeiro,800.00,16/01/2024
3,Carla Dias,Belo Horizonte,2200.75,17/01/2024
4,Diego Nunes,Porto Alegre,450.00,18/01/2024
5,Eva Melo,Recife,,19/01/2024



**📝 O que essa célula faz:**

Lê o conteúdo bruto (como texto puro) do arquivo CSV criado antes.

```python
with open('dados/clientes.csv', 'r', encoding='utf-8') as f:
    conteudo = f.read()
```

**📖 `'r'`** → modo **leitura** (read), diferente do `'w'` (escrita) usado nas células anteriores. Tentar escrever em um arquivo aberto com `'r'` geraria um erro.

**📄 `f.read()`** → lê o **conteúdo inteiro** do arquivo de uma vez só e devolve como um único texto (string). Nesse caso, o texto inclui as quebras de linha do CSV original, então ao imprimir, ele aparece formatado como o arquivo original — mas é só um texto puro, o Python ainda não entende que é uma tabela com colunas e linhas. Para isso, é necessário usar o módulo `csv`, mostrado na próxima célula.

In [ ]:
# Lendo CSV com o módulo csv — a forma correta
import csv

registros = []
with open('dados/clientes.csv', 'r', encoding='utf-8') as f:
    leitor = csv.DictReader(f)  # DictReader: cada linha vira um dicionário
    for linha in leitor:
        registros.append(linha)

print(f'Total de registros: {len(registros)}')
print('Primeiro registro:', registros[0])
print('Chaves disponíveis:', list(registros[0].keys()))

**📝 O que essa célula faz:**

Lê o mesmo arquivo CSV, mas agora de forma **estruturada**, usando o módulo `csv` do Python — transformando cada linha em um dicionário.

```python
import csv

registros = []
with open('dados/clientes.csv', 'r', encoding='utf-8') as f:
    leitor = csv.DictReader(f)
    for linha in leitor:
        registros.append(linha)
```

**🧰 `csv.DictReader(f)`** → cria um "leitor" que entende a estrutura do CSV: ele automaticamente usa a **primeira linha** do arquivo (o cabeçalho: `id,nome,cidade,valor,data`) como as **chaves** de cada dicionário, e cada linha seguinte se transforma em um dicionário com esses campos preenchidos.

**🔁 `for linha in leitor:`** → percorre o leitor linha por linha. Cada `linha` já vem como um dicionário, do tipo `{'id': '1', 'nome': 'Ana Lima', 'cidade': 'São Paulo', ...}`.

**⚠️ Detalhe importante: tudo vem como texto!** Mesmo o campo `id` e `valor`, que parecem números, chegam como **texto** (`'1'`, `'1500.50'`), porque CSV não tem conceito de tipos — é tudo texto puro separado por vírgulas. Você precisaria converter manualmente para `int`/`float` se quisesse fazer contas com esses valores (como vimos nas funções da Aula 4).

In [ ]:
# Inspecionando o que chegou
for rec in registros:
    print(f"id={rec['id']} | nome={rec['nome']:<20} | valor={rec['valor'] or 'VAZIO'}")

**📝 O que essa célula faz:**

Percorre a lista de registros lidos do CSV e imprime cada um formatado, tratando o caso de um campo vazio.

```python
for rec in registros:
    print(f"id={rec['id']} | nome={rec['nome']:<20} | valor={rec['valor'] or 'VAZIO'}")
```

**📐 `{rec['nome']:<20}`** → reserva 20 caracteres de largura para o nome, alinhado à **esquerda** (o `<` indica a direção), deixando as colunas visualmente alinhadas na saída.

**🔍 `rec['valor'] or 'VAZIO'` — um truque comum em Python:**

O operador `or` aqui não está fazendo uma comparação lógica clássica — está sendo usado como um "valor padrão rápido". Se `rec['valor']` for um texto vazio (`''`), o Python considera isso "falso" (vazio é tratado como falso em testes lógicos), então a expressão usa o valor à direita do `or` (`'VAZIO'`) no lugar. Se `rec['valor']` tiver algum conteúdo, esse conteúdo é usado normalmente.

Esse é exatamente o caso da Eva Melo no CSV de teste, cujo campo `valor` ficou vazio — ela aparecerá como `valor=VAZIO` em vez de um valor em branco confuso.

## 3. Lendo e escrevendo JSON

In [ ]:
import json

# Lendo JSON
with open('dados/pipeline_config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

print('Pipeline:', config['pipeline'])
print('Batch size:', config['configuracao']['batch_size'])
print('Tags:', config['tags'])

**📝 O que essa célula faz:**

Lê o arquivo JSON salvo anteriormente e acessa campos dele, inclusive campos **aninhados** (dentro de outro dicionário).

```python
with open('dados/pipeline_config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

print('Batch size:', config['configuracao']['batch_size'])
```

**📥 `json.load(f)`** → lê o conteúdo do arquivo JSON e o converte automaticamente em um dicionário Python (o oposto do `json.dump()` que vimos antes, que faz a conversão inversa: dicionário → JSON).

**🪆 `config['configuracao']['batch_size']` — acesso encadeado:**

Como `'configuracao'` dentro do JSON é, ele mesmo, **outro dicionário** (`{'batch_size': 10000, 'timeout_segundos': 300, ...}`), você precisa de **dois** acessos por chave em sequência: primeiro entra no dicionário `'configuracao'`, depois pega o campo `'batch_size'` de dentro dele. Isso é igual a navegar pastas dentro de pastas — cada `[chave]` é um nível mais profundo.

In [ ]:
# Escrevendo JSON
resultado_execucao = {
    'pipeline': config['pipeline'],
    'status': 'sucesso',
    'registros_processados': 4,
    'registros_rejeitados': 1,
    'data_execucao': '2024-01-19T10:30:00'
}

with open('dados/resultado_execucao.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_execucao, f, ensure_ascii=False, indent=2)

print('Resultado salvo em: dados/resultado_execucao.json')

# Verificando
with open('dados/resultado_execucao.json') as f:
    print(f.read())

**📝 O que essa célula faz:**

Cria um novo dicionário com o resultado de uma execução simulada de pipeline, e o salva como um novo arquivo JSON.

```python
resultado_execucao = {
    'pipeline': config['pipeline'],
    'status': 'sucesso',
    ...
}

with open('dados/resultado_execucao.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_execucao, f, ensure_ascii=False, indent=2)
```

**🔄 `config['pipeline']`** → reaproveita o nome do pipeline lido do arquivo de configuração anterior, em vez de digitar esse nome de novo. É uma boa prática: usar a informação que já existe em vez de duplicá-la manualmente.

**💾 O resto é o mesmo padrão já visto:** abrir com `'w'`, usar `json.dump()` com `ensure_ascii=False` e `indent=2` para salvar um JSON legível.

**🔍 A última parte da célula reabre o arquivo recém-criado** só para confirmar visualmente que ele foi salvo corretamente — uma boa prática de "confiar, mas verificar" depois de uma operação de escrita.

## 4. O padrão `with open` — por que usamos assim?

O `with open(...)` garante que o arquivo é **fechado automaticamente** ao sair do bloco, mesmo se ocorrer um erro. Sempre use esse padrão.

In [ ]:
# Modos de abertura mais comuns:
# 'r' = leitura (padrão)
# 'w' = escrita (sobrescreve se existir)
# 'a' = append (adiciona ao final)
# 'r+' = leitura e escrita

# Exemplo: adicionar linha a um log
with open('dados/log.txt', 'a', encoding='utf-8') as f:
    f.write('2024-01-19 10:30:00 | INFO | Pipeline iniciado\n')
    f.write('2024-01-19 10:30:05 | INFO | Lendo arquivo de entrada\n')
    f.write('2024-01-19 10:30:10 | INFO | 4 registros processados\n')

with open('dados/log.txt', 'r') as f:
    print(f.read())

**📝 O que essa célula faz:**

Demonstra o modo de abertura **`'a'` (append)**, que adiciona conteúdo ao **final** de um arquivo já existente, sem apagar o que já estava lá.

```python
with open('dados/log.txt', 'a', encoding='utf-8') as f:
    f.write('2024-01-19 10:30:00 | INFO | Pipeline iniciado\n')
    ...
```

**📌 Diferença entre os modos de abertura:**

| Modo | O que faz |
|---|---|
| `'r'` | lê (erro se o arquivo não existir) |
| `'w'` | escreve, **sobrescrevendo** tudo que já existia |
| `'a'` | escreve, **adicionando** ao final, sem apagar o conteúdo anterior |

**🔚 Por que cada `f.write()` termina com `\n`?**

`\n` é o caractere especial de "quebra de linha". Sem ele, as três chamadas de `f.write()` ficariam todas coladas em uma única linha no arquivo. Com `\n` no final de cada uma, cada `write` vira uma linha separada — exatamente como um arquivo de log real, onde cada linha é um evento.

## 5. Verificando se um arquivo existe

Antes de ler, verifique se o arquivo está lá:

In [ ]:
import os

arquivo = 'dados/clientes.csv'

if os.path.exists(arquivo):
    print(f'Arquivo encontrado: {arquivo}')
    tamanho = os.path.getsize(arquivo)
    print(f'Tamanho: {tamanho} bytes ({tamanho/1024:.1f} KB)')
else:
    print(f'Arquivo não encontrado: {arquivo}')

**📝 O que essa célula faz:**

Verifica se um arquivo existe **antes** de tentar fazer qualquer coisa com ele, e consulta o tamanho do arquivo em bytes.

```python
if os.path.exists(arquivo):
    tamanho = os.path.getsize(arquivo)
    print(f'Tamanho: {tamanho} bytes ({tamanho/1024:.1f} KB)')
else:
    print(f'Arquivo não encontrado: {arquivo}')
```

**✅ `os.path.exists(caminho)`** → devolve `True` se o arquivo (ou pasta) existir naquele caminho, `False` caso contrário. É uma boa prática **sempre** checar isso antes de tentar abrir um arquivo — caso contrário, tentar abrir um arquivo que não existe gera um erro (`FileNotFoundError`).

**📏 `os.path.getsize(caminho)`** → devolve o tamanho do arquivo em **bytes**. A divisão por `1024` na f-string converte esse valor para **KB** (kilobytes), seguindo a mesma lógica de potências de 2 vista na Aula 1.

**Por que isso importa em dados?** Antes de processar um arquivo grande, é comum verificar seu tamanho — para decidir, por exemplo, se ele deve ser lido de uma vez ou em pedaços (chunks), ou para alertar se um arquivo esperado está vazio ou ausente.

---

## 🏋️ Exercício — Ler CSV, processar e salvar JSON

Este exercício conecta tudo que aprendemos:

1. Ler o arquivo `dados/clientes.csv`
2. Para cada registro:
   - Converter `valor` para `float` (campo pode estar vazio — use `0.0` como padrão)
   - Converter `data` do formato `dd/mm/aaaa` para `aaaa-mm-dd`
   - Classificar o cliente: `'premium'` se valor > 1000, `'standard'` caso contrário
3. Salvar o resultado em `dados/clientes_processados.json`
4. Imprimir um resumo: total de clientes, premium vs standard

In [ ]:
import csv, json

# --- Funções auxiliares (reutilize as da Aula 4 se quiser) ---

def converter_valor(texto):
    """Converte texto de valor para float. Retorna 0.0 se vazio."""
    pass

def converter_data(texto):
    """Converte 'dd/mm/aaaa' para 'aaaa-mm-dd'."""
    pass

def classificar_cliente(valor):
    """Retorna 'premium' ou 'standard'."""
    pass


# --- Pipeline principal ---

# 1. Ler CSV

# 2. Processar cada registro

# 3. Salvar JSON

# 4. Resumo


---

## Resumo da aula

| Conceito | Sintaxe |
|---|---|
| Abrir arquivo para leitura | `with open('arquivo.txt', 'r') as f:` |
| Abrir para escrita | `with open('arquivo.txt', 'w') as f:` |
| Ler CSV como dicionários | `csv.DictReader(f)` |
| Ler JSON | `json.load(f)` |
| Salvar JSON | `json.dump(dado, f, indent=2)` |
| Verificar existência | `os.path.exists(caminho)` |

**Próxima aula:** mini-pipeline completo — juntando tudo que aprendemos.